# 2장 2강: 웹 크롤링과 수집 방법 선택 — 실습문제


## 실습 목표

- 정적·동적 크롤링의 차이를 초기 HTML과 JavaScript 실행 여부로 설명할 수 있다.
- Requests로 받은 HTML에서 `div.quote`를 찾아 필요한 요소의 존재 여부를 확인할 수 있다.
- CSS 선택자로 같은 명언 영역의 본문·작성자·태그를 추출할 수 있다.
- CSS 선택자와 XPath의 목적·표현 방식의 차이를 설명할 수 있다.
- Next 링크의 `href`와 `urljoin()`으로 다음 페이지를 요청할 수 있다.
- 여러 페이지의 레코드를 DataFrame과 CSV로 저장하고 다시 읽어 기본 구조를 확인할 수 있다.
- 필요한 데이터의 제공 방식에 따라 API·정적 크롤링·동적 크롤링 검토 순서를 설명할 수 있다.

## 실습 환경 / 데이터

- 기준 교안: **2장 2강: 웹 크롤링과 수집 방법 선택**
- 환경: Python 3, Jupyter Notebook 또는 Google Colab, 인터넷 연결
- 라이브러리: `requests`, `beautifulsoup4`, `pandas`, Python 기본 모듈 `time`, `urllib.parse`
- 일반 HTML 페이지: [Quotes to Scrape](https://quotes.toscrape.com/)
- JavaScript 기반 페이지: [Quotes to Scrape JS](https://quotes.toscrape.com/js/)
- 입력 CSV는 필요하지 않습니다. 실제 웹페이지의 HTML을 요청하여 값을 추출합니다.

| 문제 | 실습 내용 | 결과물 |
|---|---|---|
| 필수 1 | 일반·JavaScript 페이지의 초기 HTML 비교, CSS 선택자와 XPath 이해 | 실행 결과·관찰표·해석 답변 |
| 필수 2 | 일반 HTML의 Next 링크를 따라 전체 수집·저장 | `quotes_html_raw.csv` |
| 과제 1 | 2페이지 한 페이지만 추출·저장, 상황별 수집 방법 선택 | `quotes_html_page2.csv`, 선택 근거 |

**공통 컬럼**

| 컬럼 | 내용 | HTML에서 가져올 위치 |
|---|---|---|
| `text` | 명언 본문 | 각 명언 영역의 `span.text` |
| `author` | 작성자 이름 | 같은 영역의 `small.author` |
| `tags` | 관련 태그를 쉼표와 공백으로 연결한 문자열 | 같은 영역의 `div.tags a.tag` |
| `source_url` | 수집한 페이지의 주소 | 해당 요청의 `response.url` |

한 행은 명언 하나입니다. 동적 크롤링과 XPath는 교안처럼 **개념과 선택 기준만** 다룹니다. Selenium·Playwright·XPath 실행 코드를 작성하지 않습니다. 중복·결측 처리, 자동 재시도, DB 저장도 이번 요구사항에 포함하지 않습니다.


## 시작 전에 읽을 기초 개념

2장 1강에서는 JSON 응답에서 Key를 따라 명언을 꺼냈습니다. 이번에는 **웹페이지의 HTML에서 명언 영역을 찾고, 그 안의 글자를 꺼냅니다.** 데이터가 화면에 나타나는 방식까지 살펴보며 어떤 수집 방법이 적절한지 판단합니다.

### 용어를 이번 예시에 연결하기

| 용어 | 쉽게 풀어쓴 뜻 | 이번 실습에서의 예 |
|---|---|---|
| 웹 크롤링 | 웹페이지에 접근하여 필요한 데이터를 자동으로 수집하는 과정 | 명언 페이지를 요청하고 본문·작성자 추출 |
| 초기 HTML | 브라우저가 JavaScript로 화면을 바꾸기 전, 서버에서 받은 HTML | Requests가 받은 response.text |
| 정적 크롤링 | 받은 HTML에 있는 필요한 데이터를 직접 추출하는 방법 | Requests와 BeautifulSoup으로 일반 페이지 처리 |
| 동적 크롤링 | JavaScript 실행이나 사용자 동작 후의 결과를 활용하는 방법 | 브라우저 실행 결과가 필요할 때 검토 |
| JavaScript | 브라우저에서 실행되며 요소를 추가하거나 화면 등을 바꿀 수 있는 코드 | JS 예시 페이지에서 명언 요소 표시 |
| HTML 요소 | 태그와 속성, 내용 등으로 구성된 문서의 단위 | 명언 한 개를 둘러싼 div 요소 |
| class | HTML 요소를 구분하는 속성 | 명언 영역의 quote, 본문 요소의 text |
| CSS 선택자 | 원하는 HTML 요소를 찾는 조건 표현 | div.quote, span.text |
| XPath | 문서의 경로와 조건으로 요소를 찾는 표현 | 교안의 본문 요소 탐색 표현 |
| 파싱 | HTML 등의 구조를 프로그램이 다룰 수 있게 해석하는 일 | BeautifulSoup으로 받은 HTML 해석 |
| None | 값이나 대상이 없음을 나타내는 Python 값 | select_one()으로 요소를 찾지 못한 결과 |
| href | 링크가 가리키는 주소를 담는 HTML 속성 | Next 링크의 /page/2/ |
| 상대경로 | 기준 주소와 함께 해석하는 경로 | 도메인 없이 적힌 /page/2/ |
| 전체 URL | 통신 방식·서버 주소 등을 포함한 요청 주소 | https://quotes.toscrape.com/page/2/ |
| 페이지네이션 | 데이터를 여러 페이지에 나누어 제공하는 방식 | Next 링크로 다음 명언 페이지 이동 |
| 레코드 / 필드 | 명언 하나의 정보 묶음 / 그 안의 항목 | dict 하나 / 본문·작성자·태그·출처 |

**‘정적’은 데이터가 영원히 바뀌지 않는다는 뜻이 아닙니다.** 여기서는 필요한 값이 요청으로 받은 HTML에 들어 있는지, 브라우저 실행 후 결과가 필요한지를 구분합니다. 사이트가 JavaScript를 사용하더라도 원하는 값이 초기 HTML에 있다면 정적으로 추출할 수 있습니다.

### 도구의 역할을 나누어 보기

| 도구·코드 | 맡은 일 | 이 동작만으로 되지 않는 일 |
|---|---|---|
| requests.get() | 서버에 요청하고 응답을 받음 | 브라우저처럼 JavaScript 실행 |
| response.text | 받은 응답 본문을 문자열로 사용 | 필요한 명언만 자동 추출 |
| BeautifulSoup() | HTML의 구조를 해석 | JavaScript로 새 화면 요소 생성 |
| select()·select_one() | 지정한 조건의 요소 찾기 | 아직 요소 안의 글자만 얻은 것은 아님 |
| get_text() | 요소에서 실제 텍스트 얻기 | 다음 페이지로 이동 |
| urljoin() | 기준 주소와 링크 경로로 URL 구성 | 그 주소에 실제 요청하기 |
| DataFrame·to_csv() | 레코드를 표로 정리하고 파일로 기록 | 모든 품질 문제의 해결 |

**HTML을 받는 일 → 구조를 해석하는 일 → 요소를 찾는 일 → 글자를 꺼내는 일**을 나누어 읽으세요. 각 단계에서 사용하는 값의 종류가 다릅니다.

### 전체 실습 흐름

| 문제 | 먼저 확인하는 것 | 수행 과정 | 마지막에 설명할 것 |
|---|---|---|---|
| 필수 1 | 두 페이지의 초기 HTML | 요청 → 파싱 → 명언 영역 수·첫 본문 확인 | 0개라는 관찰과 요청 실패의 차이 |
| 필수 2 | 일반 HTML의 명언 영역과 Next 링크 | 영역별 추출 → 누적 → 다음 링크 이동 → CSV | 종료 근거·전체 건수·저장 결과 |
| 과제 1 | 일반 HTML의 2페이지 | 한 페이지 추출 → 별도 CSV → 결과 확인 | 수집 범위와 상황별 방법 선택 이유 |

### 코드를 읽을 때 구분할 값과 기호

| 표현 | 읽는 방법 |
|---|---|
| `response`, `soup`, `quotes` | 응답 객체 / 해석한 HTML / 선택된 명언 요소 목록 |
| `quote` | 반복 중인 명언 요소 하나 |
| `text_element` | 명언 본문을 담은 HTML 요소; 아직 본문 문자열이 아님 |
| `html_records` | 필요한 네 필드로 정리한 명언별 dict 목록 |
| `if quotes:` / `if not quotes:` | 목록에 항목이 있으면 / 목록이 비어 있으면 |
| `is None` / `is not None` | 대상이 없으면 / 대상이 있으면 |
| `A or B` | 둘 중 하나라도 참이면 |
| `page_count += 1` | 처리한 페이지 수를 1 늘림 |
| `raise ValueError(...)` | 정해 둔 문제 상황을 예외로 알려 실행을 중단 |

이번 코드의 문자열, 목록, 요소, None을 구분해야 조건문의 의미가 보입니다. 들여쓰기는 명언마다 반복할 부분, 페이지마다 반복할 부분, 반복 후 저장할 부분을 나누는 역할을 합니다.


## 실습 준비

1. `requests`, `beautifulsoup4`, `pandas`가 설치된 환경에서 아래 제공 셀을 실행하세요.
2. 일반 페이지와 JavaScript 기반 페이지 주소를 확인하세요. 준비 셀은 요청을 보내지 않습니다.
3. 필수 1 → 필수 2 → 과제 1 순서로 진행하세요. 각 문제는 준비 셀을 실행한 상태에서 별도의 결과 변수로 시작합니다.
4. 실시간 페이지의 건수나 HTML 구조는 바뀔 수 있습니다. 실제 출력과 교안의 설명을 비교하여 답하세요.

**실행 안내**

- Requests는 브라우저처럼 JavaScript를 실행하지 않습니다. `div.quote`가 0개라는 결과를 해석할 때 HTTP 요청 실패와 요소를 찾지 못한 상황을 구분하세요.
- 교안처럼 사이트의 이용 조건·접근 정책·요청 제한을 확인하고 필요한 요청만 수행합니다. 오류를 만들기 위한 반복 요청은 하지 않습니다.
- 필수 2는 최대 20페이지, 다음 페이지 요청 전 0.5초 간격을 사용합니다.
- 필수 2와 과제의 수집·저장 코드는 한 셀 안에 작성하세요. 교안처럼 요청·추출 중 예외가 발생하면 해당 셀의 실행이 멈춰 아래 저장 단계로 진행하지 않도록 합니다.
- 오류가 발생했을 때 같은 이름의 예전 CSV가 남아 있을 수 있습니다. 그 파일의 존재만으로 이번 수집 성공을 판단하지 마세요.
- CSV는 현재 작업 폴더에 저장됩니다. 같은 파일명으로 다시 정상 저장하면 기존 내용이 갱신됩니다.


In [1]:
import time
from urllib.parse import urljoin
import requests
import pandas as pd
from bs4 import BeautifulSoup

HTML_URL = "https://quotes.toscrape.com/"
JS_URL = "https://quotes.toscrape.com/js/"
COLUMNS = ["text", "author", "tags", "source_url"]

print("일반 HTML 페이지:", HTML_URL)
print("JavaScript 기반 페이지:", JS_URL)
print("최종 컬럼:", COLUMNS)
print("준비 완료: 아직 웹페이지 요청은 보내지 않았습니다.")


일반 HTML 페이지: https://quotes.toscrape.com/
JavaScript 기반 페이지: https://quotes.toscrape.com/js/
최종 컬럼: ['text', 'author', 'tags', 'source_url']
준비 완료: 아직 웹페이지 요청은 보내지 않았습니다.


### 준비 코드 상세 해설

| 코드·설정 | 의미 | 준비 직후의 상태 |
|---|---|---|
| `import time` | 다음 요청 전 대기에 사용할 도구 | 이후 time.sleep() 사용 가능 |
| `from urllib.parse import urljoin` | URL을 구성하는 함수 가져오기 | 링크 주소 결합에 사용 가능 |
| `import requests` | HTTP 요청 도구 준비 | 아직 웹 요청 없음 |
| `import pandas as pd` | 표·CSV 도구를 pd라는 이름으로 사용 | 아직 표·파일 없음 |
| `from bs4 import BeautifulSoup` | HTML 해석 도구 준비 | 아직 HTML 파싱 없음 |
| `HTML_URL` | 일반 페이지의 시작 주소 | 문자열 |
| `JS_URL` | 비교할 JavaScript 페이지 주소 | 문자열 |
| `COLUMNS` | 저장할 네 필드의 이름과 순서 | 문자열 네 개를 담은 list |
| 마지막 print()들 | 설정한 주소·컬럼과 준비 문구 출력 | 접속 성공을 확인한 출력은 아님 |

이번 세 문제는 준비 셀의 설정을 사용하지만 결과는 문제별로 새로 만듭니다. API 수집 파일을 먼저 만들어 둘 필요가 없습니다. 필수 2의 전체 결과는 quotes_html_raw.csv, 과제의 한 페이지 결과는 quotes_html_page2.csv에 저장합니다.

### 공통 필드를 HTML의 위치와 연결하기

| 최종 필드 | 값을 찾는 위치 | 저장할 형태 |
|---|---|---|
| text | 현재 div.quote 안의 span.text | 본문 문자열 |
| author | 같은 div.quote 안의 small.author | 이름 문자열 |
| tags | 같은 영역의 div.tags 아래 a.tag 요소들 | 태그를 쉼표·공백으로 연결한 문자열 |
| source_url | 현재 response.url | 해당 페이지 주소 문자열 |

한 명언의 영역을 먼저 고르는 이유는 **서로 관련된 값들을 같은 레코드로 묶기 위해서**입니다. 명언 하나에 태그가 여러 개 있어도 태그를 한 칸에 넣으므로 한 명언은 한 행으로 유지됩니다.


---

## 필수 1. 초기 HTML 비교와 수집 방식 판단

### 문제 1-1. 화면에 보이는 데이터와 초기 HTML의 요소 구분하기

#### 문제 설명

교안의 일반 페이지와 JavaScript 기반 페이지를 Requests로 요청하고, 초기 HTML의 명언 영역을 비교하세요. 브라우저 자동화를 구현하는 문제가 아니라, 같은 선택자로 찾은 요소 수와 요청 상태를 근거로 수집 방식을 판단하는 문제입니다.

#### 요구사항

1. `pages` Dictionary에 일반 페이지와 JavaScript 기반 페이지의 이름·주소를 넣고 `for name, url in pages.items()`로 순회하세요.
2. 각 페이지의 `try` 안에서 `requests.get(url, timeout=10)`과 `raise_for_status()`를 사용하세요. 요청 URL과 상태 코드를 출력하세요. 요청 오류는 `except requests.RequestException`으로 안내하세요.
3. `BeautifulSoup(response.text, "html.parser")`로 `soup`을 만들고 `soup.select("div.quote")` 결과를 `quotes`에 저장하세요. 초기 HTML에서 찾은 명언 영역 수를 출력하세요.
4. 명언 영역이 있으면 첫 영역에서 `span.text`를 `select_one()`으로 찾으세요. 요소가 `None`이 아닐 때만 `get_text(strip=True)`로 첫 본문을 출력하세요. 명언 영역이 없으면 찾지 못했다는 안내를 출력하세요.
5. 관찰표에 실제 요청 상태와 요소 수를 적고, 선택자 표에 각 표현의 의미를 적으세요. Q1~Q5에 답하세요. 사이트가 달라졌거나 요청이 실패했다면 해당 상황을 기록하세요.

#### 해석 질문

**Q1.** 일반 HTML 페이지는 어떤 조건에서 Requests와 BeautifulSoup으로 수집할 수 있나요? JavaScript 기반 페이지와 어떤 차이가 있나요?

**Q2.** 상태 코드 200인데 `div.quote`가 0개라면 반드시 요청 실패인가요? 웹페이지 전체에 명언 관련 데이터가 전혀 없다고 단정해도 되나요?

**Q3.** `select()`와 `select_one()`의 차이는 무엇이며, 첫 본문 요소가 `None`인지 확인하는 이유는 무엇인가요?

**Q4.** CSS `div.quote span.text`와 교안의 XPath `//div[@class='quote']//span[@class='text']`는 무엇을 찾으려는 표현인가요? BeautifulSoup의 `select()`에 XPath를 그대로 넣어도 되나요?

**Q5.** JavaScript를 사용하는 페이지라면 반드시 Selenium이나 Playwright가 필요한가요? 브라우저 자동화를 검토하기 전에 무엇을 확인해야 하나요?

#### 제출 결과

- 두 페이지 요청·HTML 확인 코드와 출력
- 관찰표·선택자 표
- Q1~Q5 답변


### 작성 도움말 — 필수 1

이 문제는 전체 명언 파일을 만드는 작업보다 **두 페이지의 초기 HTML을 관찰하고 결과를 구분하는 작업**에 집중합니다.

- 이름·주소 쌍을 하나씩 받아 같은 요청·파싱·선택 과정을 적용하세요.
- 먼저 명언 영역 목록이 비었는지, 다음으로 첫 본문 요소가 None인지 확인합니다.
- 관찰표에는 실제 HTTP 상태와 요소 수를 별도로 적습니다. 요청이 실패했다면 요소 수를 임의로 0이라고 채우지 마세요.
- 선택자 표는 태그, 클래스, 포함 관계로 나누어 읽습니다. XPath는 의미를 비교하며 실행 코드는 추가하지 않습니다.


### 코드 작성란

아래 셀에 코드를 작성하고 실행 결과를 남기세요.


In [2]:
# 필수 1. 초기 HTML 비교와 수집 방식 판단 코드를 작성하세요.
# 위 요구사항의 변수명과 파일명을 사용하세요.

# 1. `pages` Dictionary에 일반 페이지와 JavaScript 기반 페이지의 이름·주소를 넣고 `for name, url in pages.items()`로 순회하세요.
pages = {
    "일반 HTML 페이지": HTML_URL,
    "JavaScript 기반 페이지": JS_URL,
}

for name, url in pages.items():
    print(f"\n[{name}]")
    try:
        # 2. 각 페이지의 `try` 안에서 `requests.get(url, timeout=10)`과 `raise_for_status()`를 사용하세요. 요청 URL과 상태 코드를 출력하세요.
        response = requests.get(url, timeout=10)
        print("요청 URL:", response.url)
        print("상태 코드:", response.status_code)
        response.raise_for_status()

    # 2. 요청 오류는 `except requests.RequestException`으로 안내하세요.
    except requests.RequestException as error:
        print("요청 실패:", error)

    else:
        # 3. `BeautifulSoup(response.text, "html.parser")`로 `soup`을 만들고 `soup.select("div.quote")` 결과를 `quotes`에 저장하세요. 초기 HTML에서 찾은 명언 영역 수를 출력하세요.
        soup = BeautifulSoup(response.text, "html.parser")
        quotes = soup.select("div.quote")
        print("초기 HTML의 명언 영역 수:", len(quotes))

        # 4. 명언 영역이 있으면 첫 영역에서 `span.text`를 `select_one()`으로 찾으세요. 요소가 `None`이 아닐 때만 `get_text(strip=True)`로 첫 본문을 출력하세요. 명언 영역이 없으면 찾지 못했다는 안내를 출력하세요.
        if quotes:
            first_text = quotes[0].select_one("span.text")
            if first_text is not None:
                print("첫 본문:", first_text.get_text(strip=True))
            else:
                print("첫 명언 영역에서 span.text를 찾지 못했습니다.")
        else:
            print("초기 HTML에서 div.quote를 찾지 못했습니다.")


[일반 HTML 페이지]
요청 URL: https://quotes.toscrape.com/
상태 코드: 200
초기 HTML의 명언 영역 수: 10
첫 본문: “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”

[JavaScript 기반 페이지]
요청 URL: https://quotes.toscrape.com/js/
상태 코드: 200
초기 HTML의 명언 영역 수: 0
초기 HTML에서 div.quote를 찾지 못했습니다.


### 답변 작성란


| 페이지 | 요청 상태·오류 | 초기 HTML의 명언 영역 수 | 결과 해석 |
|---|---|---|---|
| 일반 HTML 페이지 | 200 (요청 성공) | 10 | 요청이 성공했고 HTML에서 지정한 요소를 찾음 |
| JavaScript 기반 페이지 | 200 (요청 성공) | 0 | 요청은 성공했지만 초기 HTML에서 지정한 요소를 찾지 못함 |

| CSS 선택자 | 선택하는 대상 |
|---|---|
| `div.quote` | quote 클래스를 가진 div요소 = 명언 한 개를 감싸는 영역 |
| `span.text` | text 클래스를 가진 span 요소 = 명언 본문 |
| `small.author` | author 클래스를 가진 small 요소 = 작성자 이름 |
| `div.tags a.tag` | tags 클래스를 가진 div 하위에 있는, tag 클래스의 a 요소 = 관련 태그 각각 |
| `div.quote span.text` | quote 클래스를 가진 div의 하위에 있는 text 클래스의 span 요소 = 명언 영역 안의 본문 (띄어쓰기는 "안쪽"이라는 뜻) |


**Q1. 일반 HTML 페이지는 어떤 조건에서 Requests와 BeautifulSoup으로 수집할 수 있나요? JavaScript 기반 페이지와 어떤 차이가 있나요?**

- 서버가 반환한 HTML에 필요한 명언 요소가 이미 포함되어있으면 Requests로 가져와서 BeautifulSoup로 바로 추출할 수 있다.  
- JavaScript 기반 페이지는 브라우저가 JavaScript를 실행하면서 명언 요소를 만들 수 있어 초기 HTML과 실행 후 화면이 다를 수 있다.  

**Q2. 상태 코드 200인데 `div.quote`가 0개라면 반드시 요청 실패인가요? 웹페이지 전체에 명언 관련 데이터가 전혀 없다고 단정해도 되나요?**

- 답변: 상태 코드가 200(정상)인데 0개는 초기 HTML에서 div.quote를 찾지 못했다는 것을 의미  
- HTML을 다시 한 번 확인하여 응답 가능성을 재확인해야함

**Q3. `select()`와 `select_one()`의 차이는 무엇이며, 첫 본문 요소가 `None`인지 확인하는 이유는 무엇인가요?**

- 답변:  
    - **`select()`** : 일치하는 모든 요소를 목록으로 가져옴  
    - **`select_one()`** : 일치하는 첫 요소를 가져옴  
    -> select_one()은 요소를 못찾으면 None을 반환하므로 get_text()를 호출하지 않도록 존재 여부를 확인한다.  

**Q4. CSS `div.quote span.text`와 교안의 XPath `//div[@class='quote']//span[@class='text']`는 무엇을 찾으려는 표현인가요? BeautifulSoup의 `select()`에 XPath를 그대로 넣어도 되나요?**

- 두 예시는 명언 아래 본문 요소를 찾으려는 표현  
- CSS는 태그, class와 하위 관계로 표현하고 XPath는 문서 경로, 조건으로 표현한다.  
제시된 XPath의 @class 조건은 정확한 속성값 비교  
- beautifulsoup의 select()는 CSS를 선택자로 사용하므로 XPath를 그대로 넣어 실행하지 않는다. 

**Q5. JavaScript를 사용하는 페이지라면 반드시 Selenium이나 Playwright가 필요한가요? 브라우저 자동화를 검토하기 전에 무엇을 확인해야 하나요?**

- 답변: **반드시 필요한 것은 아님**  
    - 필요한 데이터가 초기 HTML에 있으면 정적 크롤링을, 접근 가능한 API나 JSON 요청이 있으면 그 활용을 먼저 검토

---

## 필수 2. Next 링크를 따라 HTML 수집·저장

### 문제 2-1. 명언별 정보를 연결하고 마지막 페이지까지 수집하기

#### 문제 설명

일반 HTML 페이지에서 명언 영역별로 본문·작성자·태그를 추출하고, Next 링크가 있으면 해당 주소로 이동하세요. 교안의 네 컬럼을 유지하여 전체 결과를 CSV로 저장한 뒤 다시 읽어 확인합니다.

#### 요구사항

1. `url = HTML_URL`, `html_records = []`, `page_count = 0`, `max_pages = 20`으로 시작하세요. `while url is not None`을 사용하고 요청 전 `page_count >= max_pages`이면 RuntimeError로 중단하세요.
2. 현재 URL에 `timeout=10`으로 요청하고 `raise_for_status()`로 확인한 뒤 HTML을 BeautifulSoup으로 파싱하세요. `div.quote` 영역이 없으면 ValueError로 중단하세요.
3. 각 영역에서 `span.text`, `small.author`, `div.tags a.tag`를 찾으세요. 본문 또는 작성자 요소가 None이면 ValueError로 중단하고, 정상인 경우 네 공통 필드를 Dictionary로 만들어 `html_records`에 추가하세요. 태그는 쉼표와 공백으로 연결하고 출처는 현재 `response.url`을 사용하세요.
4. 현재 페이지 추출을 마치면 `page_count`를 1 증가시키고 이번 페이지 건수·누적 건수를 출력하세요.
5. `soup.select_one("li.next a")`로 `next_link`를 찾으세요. 링크가 없으면 `url = None`, 있으면 `.get("href")`로 주소를 꺼내세요. href가 비어 있으면 ValueError로 중단하고, 정상이라면 `urljoin(response.url, href)`로 다음 URL을 만들고 0.5초 대기하세요.
6. 반복 정상 종료 후 같은 코드 셀에서 `html_df = pd.DataFrame(html_records, columns=COLUMNS)`를 만들고 `quotes_html_raw.csv`에 `index=False`, `encoding="utf-8-sig"`로 저장하세요.
7. 같은 파일을 `pd.read_csv(..., encoding="utf-8-sig", keep_default_na=False)`로 읽어 `html_loaded_df`에 저장하세요. 수집 페이지 수·전체 건수·저장 후 건수·컬럼·저장 파일명·`head()`를 출력하세요.
8. 결과 확인표, API·HTML 비교표와 Q1~Q5에 답하세요. 이번에는 API 파일을 불러오거나 두 파일을 합치는 코드를 작성하지 않습니다.

#### 해석 질문

**Q1.** 본문과 작성자를 따로 전체 수집하는 대신 같은 `div.quote` 영역 안에서 추출하는 이유는 무엇인가요? 요소가 None인 것과 텍스트가 빈 문자열인 것은 같은가요?

**Q2.** 다음 페이지 링크의 선택자와 주소 속성은 무엇인가요? `urljoin("https://quotes.toscrape.com/", "/page/2/")`의 결과를 적으세요.

**Q3.** 1페이지 10개·Next 있음, 2페이지 10개·Next 있음, 3페이지 5개·Next 없음이라면 몇 페이지·몇 건인가요? 마지막 페이지의 데이터는 언제 추가하나요?

**Q4.** 페이지 상한 초과, 명언 영역 없음, 필수 요소 누락, Next 링크의 href 누락이 발생하면 어떻게 되나요? 예전 CSV가 남아 있으면 이번 실행 성공인가요?

**Q5.** API의 `has_next`와 HTML의 Next 링크는 어떤 공통 역할을 하나요? API·HTML 결과를 단순히 합칠 때의 문제와 저장 후 추가 확인이 필요한 이유를 설명하세요.

#### 제출 결과

- HTML 반복 수집·저장·다시 읽기 코드와 출력
- 정상 완료한 경우 quotes_html_raw.csv
- 결과 확인표·API 비교표·Q1~Q5 답변


### 작성 도움말 — 필수 2

현재 페이지의 명언 추출을 마친 뒤 Next를 확인한다는 순서를 먼저 정리하세요.

- while은 페이지, 그 안의 for는 명언을 처리합니다. 누적 목록은 while 앞에서 한 번 만듭니다.
- 본문과 작성자는 현재 quote 안에서 찾아 같은 명언끼리 묶습니다.
- Next 요소가 없는 것과 요소는 있지만 href가 없는 것을 서로 다른 경우로 처리합니다.
- urljoin()은 주소를 만들고, 실제 다음 요청은 다음 반복에서 수행합니다.
- 수집부터 저장까지 같은 셀에 작성하여 수집 중 예외가 나면 아래 저장 줄로 진행하지 않도록 합니다.
- 실제 결과 확인표와 10·10·5개 가정 사례 계산은 구분해서 적으세요.


### 코드 작성란

아래 셀에 코드를 작성하고 실행 결과를 남기세요.


In [3]:
# 필수 2. Next 링크를 따라 HTML 수집·저장 코드를 작성하세요.
# 위 요구사항의 변수명과 파일명을 사용하세요.
from IPython.display import display

# 1. `url = HTML_URL`, `html_records = []`, `page_count = 0`, `max_pages = 20`으로 시작하세요. `while url is not None`을 사용하고 요청 전 `page_count >= max_pages`이면 RuntimeError로 중단하세요.
url = HTML_URL
html_records = []
page_count = 0
max_pages = 20

try:
    while url is not None:
        if page_count >= max_pages:
            raise RuntimeError(f"최대 {max_pages}페이지를 넘었지만 마지막 페이지에 도달하지 못했습니다.")

        # 2. 현재 URL에 `timeout=10`으로 요청하고 `raise_for_status()`로 확인한 뒤 HTML을 BeautifulSoup으로 파싱하세요. `div.quote` 영역이 없으면 ValueError로 중단하세요.
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        quotes = soup.select("div.quote")
        if not quotes:
            raise ValueError(f"{response.url}에서 div.quote 영역을 찾지 못했습니다.")

        # 3. 각 영역에서 `span.text`, `small.author`, `div.tags a.tag`를 찾으세요. 본문 또는 작성자 요소가 None이면 ValueError로 중단하고, 정상인 경우 네 공통 필드를 Dictionary로 만들어 `html_records`에 추가하세요.
        page_record_count = 0
        for quote in quotes:
            text_tag = quote.select_one("span.text")
            author_tag = quote.select_one("small.author")
            tag_tags = quote.select("div.tags a.tag")

            if text_tag is None or author_tag is None:
                raise ValueError(f"{response.url}의 명언 영역에서 본문 또는 작성자 요소가 없습니다.")

            html_records.append({
                "text": text_tag.get_text(strip=True),
                "author": author_tag.get_text(strip=True),
                "tags": ", ".join(tag.get_text(strip=True) for tag in tag_tags),
                "source_url": response.url,
            })
            page_record_count += 1

        # 4. 현재 페이지 추출을 마치면 `page_count`를 1 증가시키고 이번 페이지 건수·누적 건수를 출력하세요.
        page_count += 1
        print(f"{page_count}번째 페이지 ({response.url}) : 이번 페이지 {page_record_count}건 / 누적 {len(html_records)}건")

        # 5. `soup.select_one("li.next a")`로 `next_link`를 찾으세요. 링크가 없으면 `url = None`, 있으면 `.get("href")`로 주소를 꺼내세요. href가 비어 있으면 ValueError로 중단하고, 정상이라면 `urljoin(response.url, href)`로 다음 URL을 만들고 0.5초 대기하세요.
        next_link = soup.select_one("li.next a")
        if next_link is None:
            url = None
        else:
            href = next_link.get("href")
            if not href:
                raise ValueError(f"{response.url}의 Next 링크에 href가 없습니다.")
            url = urljoin(response.url, href)
            time.sleep(0.5)

    # 6. 반복 정상 종료 후 같은 코드 셀에서 `html_df = pd.DataFrame(html_records, columns=COLUMNS)`를 만들고 `quotes_html_raw.csv`에 `index=False`, `encoding="utf-8-sig"`로 저장하세요.
    html_df = pd.DataFrame(html_records, columns=COLUMNS)
    html_df.to_csv("quotes_html_raw.csv", index=False, encoding="utf-8-sig")

    # 7. 같은 파일을 `pd.read_csv(..., encoding="utf-8-sig", keep_default_na=False)`로 읽어 `html_loaded_df`에 저장하세요. 수집 페이지 수·전체 건수·저장 후 건수·컬럼·저장 파일명·`head()`를 출력하세요.
    html_loaded_df = pd.read_csv("quotes_html_raw.csv", encoding="utf-8-sig", keep_default_na=False)
    print("\n수집 페이지 수:", page_count)
    print("전체 건수:", len(html_records))
    print("저장 후 건수:", len(html_loaded_df))
    print("컬럼 목록:", html_loaded_df.columns.tolist())
    print("저장 파일명: quotes_html_raw.csv")
    display(html_loaded_df.head())

except (requests.RequestException, ValueError, RuntimeError) as error:
    print(f"\n수집 중단 ({page_count}페이지까지 처리):", error)
    print("이번 실행은 전체 수집을 완료하지 못해 CSV로 저장하지 않았습니다.")

1번째 페이지 (https://quotes.toscrape.com/) : 이번 페이지 10건 / 누적 10건
2번째 페이지 (https://quotes.toscrape.com/page/2/) : 이번 페이지 10건 / 누적 20건
3번째 페이지 (https://quotes.toscrape.com/page/3/) : 이번 페이지 10건 / 누적 30건
4번째 페이지 (https://quotes.toscrape.com/page/4/) : 이번 페이지 10건 / 누적 40건
5번째 페이지 (https://quotes.toscrape.com/page/5/) : 이번 페이지 10건 / 누적 50건
6번째 페이지 (https://quotes.toscrape.com/page/6/) : 이번 페이지 10건 / 누적 60건
7번째 페이지 (https://quotes.toscrape.com/page/7/) : 이번 페이지 10건 / 누적 70건
8번째 페이지 (https://quotes.toscrape.com/page/8/) : 이번 페이지 10건 / 누적 80건
9번째 페이지 (https://quotes.toscrape.com/page/9/) : 이번 페이지 10건 / 누적 90건
10번째 페이지 (https://quotes.toscrape.com/page/10/) : 이번 페이지 10건 / 누적 100건

수집 페이지 수: 10
전체 건수: 100
저장 후 건수: 100
컬럼 목록: ['text', 'author', 'tags', 'source_url']
저장 파일명: quotes_html_raw.csv


,text,author,tags,source_url
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world",https://quotes.toscrape.com/
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices",https://quotes.toscrape.com/
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles",https://quotes.toscrape.com/
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor",https://quotes.toscrape.com/
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational",https://quotes.toscrape.com/


### 답변 작성란


| 확인 항목 | 실제 실행 결과·오류 상황 | 해석 |
|---|---|---|
| 수집한 페이지 수 | 10 | 추출을 완료한 HTML 페이지 수 |
| 정상 종료 당시 Next 링크 | 없음 (10번째 페이지에서 `next_link`가 None) | Next_link is None, url은 None로 설정됨 |
| 전체 명언 수 / 저장 후 명언 수 | 100 / 100 | 누적 레코드 수와 다시 읽은 행 수가 맞는지 확인 |
| 컬럼 목록 | ['text', 'author', 'tags', 'source_url'] | 공통 네 컬럼이 COLUMNS 순서대로 저장됨 |
| 저장 파일 / 다시 읽기 결과 | quotes_html_raw.csv / 100행, head()로 첫 5행 확인 | 같은 영역에서 가져온 본문, 작성자, 태그와 출처 확인 |

가정 사례의 요청 페이지 수 / 최종 건수: 3페이지 / 25건  

| 비교 항목 | API 수집 | HTML 크롤링 |
|---|---|---|
| 응답 형식 | JSON (application/json) | HTML |
| 데이터 접근 방법 | `response.json()`으로 파싱한 뒤 Key로 접근 (`quote["author"]["name"]`) | BeautifulSoup와 CSS 선택자 |
| 다음 페이지 판단 | 응답의 `has_next` 값 (True/False) | `li.next a` , `href` |
| `source_url`의 종류 | API 주소 + Query Parameter (`…/api/quotes?page=1`) | HTML 페이지 URL |
| 최종 컬럼 | text, author, tags, source_url | text, author, tags, source_url |


**Q1. 본문과 작성자를 따로 전체 수집하는 대신 같은 `div.quote` 영역 안에서 추출하는 이유는 무엇인가요? 요소가 None인 것과 텍스트가 빈 문자열인 것은 같은가요?**

- 명언 영역 안에서 본문, 작성자를 함께 추출하면 같은 명언에 속한 정보를 하나의 레코드로 연결할 수 있다.  
- 요소가 None이면 해당 태그를 찾지 못했다는 뜻이고, 빈문자열이면 요소는 있지만 추출된 텍스트가 비어있는 경우

**Q2. 다음 페이지 링크의 선택자와 주소 속성은 무엇인가요? `urljoin("https://quotes.toscrape.com/", "/page/2/")`의 결과를 적으세요.**

- 답변:  
    - **선택자** : `li.next a` (class가 next인 li 안의 a 태그)  
    - **주소 속성** : `href`  
    - **urljoin 결과** : `https://quotes.toscrape.com/page/2/`  
        -> href는 `/page/2/`만으로는 도메인이 없기 때문에 join하여 요청할 전체 url을 만든다.  

**Q3. 1페이지 10개·Next 있음, 2페이지 10개·Next 있음, 3페이지 5개·Next 없음이라면 몇 페이지·몇 건인가요? 마지막 페이지의 데이터는 언제 추가하나요?**

- 답변: **3페이지, 25건 (10 + 10 + 5)**  
    -> 3페이지의 5개를 먼저 누적한 뒤 Next 링크가 없음을 확인하여 종료한다.  

**Q4. 페이지 상한 초과, 명언 영역 없음, 필수 요소 누락, Next 링크의 href 누락이 발생하면 어떻게 되나요? 예전 CSV가 남아 있으면 이번 실행 성공인가요?**

- 교안처럼 예외를 발생시켜 현재 셀의 실행을 중단하고 아래 저장 코드로 진행하지 않는다.  
해당 실행에서 쌓인 일부 레코드가 남을 수 있지만 전체 완료는 아니다.  
-> 예전 CSV가 남아있더라도 이번 실행 성공을 증명하지는 않음

**Q5. API의 `has_next`와 HTML의 Next 링크는 어떤 공통 역할을 하나요? API·HTML 결과를 단순히 합칠 때의 문제와 저장 후 추가 확인이 필요한 이유를 설명하세요.**

- 둘 다 다음 페이지가 있는지 판단하는데 쓰이는 단서  
같은 명언을 각각 수집했다면 

---

## 과제 1. 다른 HTML 페이지에 적용하고 수집 방법 선택

### 문제 3-1. 두 번째 페이지의 명언을 별도로 저장하기

#### 문제 설명

교안의 Next 링크 예시 `/page/2/`를 사용하여 두 번째 페이지의 전체 URL을 만들고, 해당 페이지의 명언만 추출·저장하세요. 필수 2에서 배운 선택자·필드 구성·저장 방법을 그대로 적용하며, 전체 페이지를 다시 순회하지 않습니다.

#### 요구사항

1. `assignment_url = urljoin(HTML_URL, "/page/2/")`, `assignment_records = []`를 준비하고 완성된 URL을 출력하세요.
2. 해당 URL에 `timeout=10`으로 요청하여 `assignment_response`에 저장하고 HTTP 오류를 확인하세요. HTML을 `assignment_soup`으로 파싱하여 명언 영역 `assignment_quotes`를 찾으세요. 명언 영역이 없으면 ValueError로 중단하세요.
3. 각 명언 영역에서 필수 2와 같은 선택자를 사용하세요. 본문·작성자 요소의 존재를 확인한 뒤 네 필드의 레코드를 구성하여 `assignment_records`에 추가하세요.
4. `assignment_df`를 컬럼 순서 `COLUMNS`로 만들고 `quotes_html_page2.csv`에 `index=False`, `encoding="utf-8-sig"`로 저장하세요.
5. 같은 파일을 `keep_default_na=False`, 같은 인코딩으로 다시 읽어 `assignment_loaded_df`에 저장하세요. 요청 상태·HTML 영역 수·추출 건수·Shape·저장 후 행 수·컬럼·첫 작성자(`loc[0, "author"]`)·`head()`를 출력하세요. 요청부터 저장까지 같은 셀에 작성하세요.
6. 결과 확인표를 채우세요. 아래 상황별 선택표에는 우선 검토할 방법과 이유를 적고 Q1~Q4에 답하세요. API·Network 조사·동적 크롤링을 실제 코드로 구현할 필요는 없습니다.

#### 해석 질문

**Q1.** 이번 과제 결과는 사이트 전체 수집 결과인가요? 이 페이지에 Next 링크가 있어도 따라가지 않는 이유는 무엇인가요?

**Q2.** 명언 영역 수·추출 건수·저장 후 행 수를 각각 확인하는 이유는 무엇인가요? 첫 작성자만 확인하면 모든 레코드의 연결이 맞다고 할 수 있나요?

**Q3.** 상황 C에서 브라우저 자동화를 구현하기 전에 무엇을 확인해야 하나요? 상황 A에서도 API가 항상 적합하다고 할 수 있나요?

**Q4.** CSV를 저장하기 전에 사이트 정책과 요청 제한을 확인하는 이유는 무엇이며, CSV 저장 후에는 어떤 품질 문제가 남을 수 있나요?

#### 제출 결과

- 2페이지 URL 구성·추출·저장·재확인 코드와 출력
- 정상 완료한 경우 quotes_html_page2.csv
- 결과표·상황별 선택표·Q1~Q4 답변


### 작성 도움말 — 과제 1

필수 2의 영역 선택·추출·저장을 2페이지 한 페이지에 적용합니다. 페이지를 계속 이동하는 반복은 필요하지 않습니다.

- 입력 URL과 과제 레코드 목록, 표, 파일명을 순서대로 연결하세요.
- HTML 영역 수, 만든 레코드 수, 다시 읽은 행 수가 각각 무엇을 세는지 설명합니다.
- loc[0, "author"]는 이번 표의 첫 행 작성자 값을 확인하는 표현입니다. 컬럼 이름의 철자와 행 인덱스를 확인하세요.
- 상황별 선택표에는 도구 이름뿐 아니라 '필요한 데이터가 어디에 있는가'를 근거로 적습니다. 동적 도구를 실제 구현할 필요는 없습니다.


### 코드 작성란

아래 셀에 코드를 작성하고 실행 결과를 남기세요.


In [4]:
# 과제 1. 다른 HTML 페이지에 적용하고 수집 방법 선택 코드를 작성하세요.
# 위 요구사항의 변수명과 파일명을 사용하세요.
from IPython.display import display

# 1. `assignment_url = urljoin(HTML_URL, "/page/2/")`, `assignment_records = []`를 준비하고 완성된 URL을 출력하세요.
assignment_url = urljoin(HTML_URL, "/page/2/")
assignment_records = []
print("완성된 URL:", assignment_url)

try:
    # 2. 해당 URL에 `timeout=10`으로 요청하여 `assignment_response`에 저장하고 HTTP 오류를 확인하세요. HTML을 `assignment_soup`으로 파싱하여 명언 영역 `assignment_quotes`를 찾으세요. 명언 영역이 없으면 ValueError로 중단하세요.
    assignment_response = requests.get(assignment_url, timeout=10)
    assignment_response.raise_for_status()
    assignment_soup = BeautifulSoup(assignment_response.text, "html.parser")

    assignment_quotes = assignment_soup.select("div.quote")
    if not assignment_quotes:
        raise ValueError(f"{assignment_response.url}에서 div.quote 영역을 찾지 못했습니다.")

    # 3. 각 명언 영역에서 필수 2와 같은 선택자를 사용하세요. 본문·작성자 요소의 존재를 확인한 뒤 네 필드의 레코드를 구성하여 `assignment_records`에 추가하세요.
    for quote in assignment_quotes:
        text_tag = quote.select_one("span.text")
        author_tag = quote.select_one("small.author")
        tag_tags = quote.select("div.tags a.tag")

        if text_tag is None or author_tag is None:
            raise ValueError("명언 영역에서 본문 또는 작성자 요소가 없습니다.")

        assignment_records.append({
            "text": text_tag.get_text(strip=True),
            "author": author_tag.get_text(strip=True),
            "tags": ", ".join(tag.get_text(strip=True) for tag in tag_tags),
            "source_url": assignment_response.url,
        })

    # 4. `assignment_df`를 컬럼 순서 `COLUMNS`로 만들고 `quotes_html_page2.csv`에 `index=False`, `encoding="utf-8-sig"`로 저장하세요.
    assignment_df = pd.DataFrame(assignment_records, columns=COLUMNS)
    assignment_df.to_csv("quotes_html_page2.csv", index=False, encoding="utf-8-sig")

    # 5. 같은 파일을 `keep_default_na=False`, 같은 인코딩으로 다시 읽어 `assignment_loaded_df`에 저장하세요. 요청 상태·HTML 영역 수·추출 건수·Shape·저장 후 행 수·컬럼·첫 작성자(`loc[0, "author"]`)·`head()`를 출력하세요. 요청부터 저장까지 같은 셀에 작성하세요.
    assignment_loaded_df = pd.read_csv("quotes_html_page2.csv", encoding="utf-8-sig", keep_default_na=False)
    print("요청 상태:", assignment_response.status_code)
    print("HTML 명언 영역 수:", len(assignment_quotes))
    print("추출 건수:", len(assignment_records))
    print("DataFrame Shape:", assignment_df.shape)
    print("저장 후 행 수:", len(assignment_loaded_df))
    print("컬럼:", assignment_loaded_df.columns.tolist())
    print("첫 작성자:", assignment_loaded_df.loc[0, "author"])
    display(assignment_loaded_df.head())

except (requests.RequestException, ValueError) as error:
    print("\n2페이지 수집 중단:", error)
    print("이번 실행은 2페이지 수집을 완료하지 못해 CSV로 저장하지 않았습니다.")

완성된 URL: https://quotes.toscrape.com/page/2/
요청 상태: 200
HTML 명언 영역 수: 10
추출 건수: 10
DataFrame Shape: (10, 4)
저장 후 행 수: 10
컬럼: ['text', 'author', 'tags', 'source_url']
첫 작성자: Marilyn Monroe


,text,author,tags,source_url
0,“This life is what you make it. No matter what...,Marilyn Monroe,"friends, heartbreak, inspirational, life, love...",https://quotes.toscrape.com/page/2/
1,“It takes a great deal of bravery to stand up ...,J.K. Rowling,"courage, friends",https://quotes.toscrape.com/page/2/
2,"“If you can't explain it to a six year old, yo...",Albert Einstein,"simplicity, understand",https://quotes.toscrape.com/page/2/
3,"“You may not be her first, her last, or her on...",Bob Marley,love,https://quotes.toscrape.com/page/2/
4,"“I like nonsense, it wakes up the brain cells....",Dr. Seuss,fantasy,https://quotes.toscrape.com/page/2/


### 답변 작성란


| 확인 항목 | 실행 결과·오류 상황 |
|---|---|
| 요청 URL / HTTP 상태 | https://quotes.toscrape.com/page/2/ / 200 |
| HTML 명언 영역 수 / 추출 건수 | 10 / 10 |
| DataFrame Shape / 저장 후 행 수 | (10, 4) / 10 |
| 컬럼 / 첫 작성자 | ['text', 'author', 'tags', 'source_url'] / Marilyn Monroe |
| 파일명 / 수집 범위 | quotes_html_page2.csv / 2페이지 한 페이지 (10건) |


| 상황 | 우선 검토할 방법 | 선택 근거 |
|---|---|---|
| A. 필요한 데이터가 JSON API로 제공되고 접근이 허용됨 | API 수집 (Requests + response.json()) | 필요한 데이터가 이미 JSON으로 구조화되어 있어서 Key로 바로 꺼낼 수 있음 |
| B. API는 없지만 초기 HTML에 필요한 데이터가 있음 | 정적 크롤링 (Requests + BeautifulSoup) | 서버가 처음 보낸 HTML에 데이터가 있으므로 CSS 선택자로 찾을 수 있음 |
| C. 초기 명언 요소는 없고 JavaScript 실행 후 명언이 표시됨 | 먼저 데이터가 오는 곳을 조사한 뒤, 필요할 때 동적 크롤링 | 초기 HTML에 데이터가 없어서 정적 크롤링으로는 찾을 수 없음 |


**Q1. 이번 과제 결과는 사이트 전체 수집 결과인가요? 이 페이지에 Next 링크가 있어도 따라가지 않는 이유는 무엇인가요?**

- 답변: 사이트 전체가 아닌 2페이지 한 페이지(10건)만 수집한 결과  
    -> Next 링크는 "다음 페이지가 있다"는 정보일 뿐, 수집 범위가 처음부터 2페이지로 정해져 있어 Next 링크가 있어도 따라가지 않음  

**Q2. 명언 영역 수·추출 건수·저장 후 행 수를 각각 확인하는 이유는 무엇인가요? 첫 작성자만 확인하면 모든 레코드의 연결이 맞다고 할 수 있나요?**


- **각각 확인하는 이유** : 세 숫자가 서로 다른 단계를 세기 때문임  
    - **명언 영역 수(10)** : HTML에서 찾은 div.quote 개수 (입력)  
    - **추출 건수(10)** : 그 영역들로 만든 레코드 개수 (추출)  
    - **저장 후 행 수(10)** : CSV를 다시 읽었을 때의 행 개수 (저장)  
-> 세 값이 모두 같아야 입력부터 저장까지 빠진 명언이 없다고 볼 수 있고, 다르면 어느 단계에서 달라졌는지 찾을 수 있음  
- **첫 작성자만으로는 알 수 없음**  
    첫 행 하나가 맞다는 것만 확인한 것이지 나머지 9행의 본문과 작성자가 제대로 짝지어졌다고 단정지을 수 없음.  
    -> head()로 여러 행을 함께 보고, 같은 영역 안에서 추출했는지 코드 구조도 확인해야 함  

**Q3. 상황 C에서 브라우저 자동화를 구현하기 전에 무엇을 확인해야 하나요? 상황 A에서도 API가 항상 적합하다고 할 수 있나요?**

- **상황 C에서 먼저 확인할 것**  
        1. 같은 데이터를 제공하는 공식 API가 있는지, 개발자 도구 Network 탭에서 JavaScript가 데이터를 JSON으로 받아 오는 요청이 있는지 등을 확인해볼 수 있다.  
- **상황 A에서도 API가 항상 적합한 것은 아님**  
        API가 필요한 필드를 모두 주지 않거나 요청 횟수 제한, 이용 조건, 비용이 수집 목적과 맞지 않을 수 있음.  
        -> 그래서 API가 있어도 제공 범위와 이용 조건을 먼저 확인해야 함  

**Q4. CSV를 저장하기 전에 사이트 정책과 요청 제한을 확인하는 이유는 무엇이며, CSV 저장 후에는 어떤 품질 문제가 남을 수 있나요?**


- **사이트 정책과 요청 제한을 확인하는 이유**  
        -> 이용약관에서 자동 수집을 막고 있거나, 수집한 데이터의 저장,활용 범위를 제한할 수 있기 때문  
        -> 요청 제한을 넘기면 서버에 부담을 주고 403, 429로 접근이 막힐 수 있으므로 수집과 저장이 허용된 범위인지 먼저 확인해야 함  
- **CSV 저장 후 남을 수 있는 품질 문제**  
        -> **중복** : 같은 명언이 여러 번 저장될 수 있음 (API,HTML 결과를 합치는 경우 등)  
        -> **결측** : 본문이나 태그가 빈 값인 행이 있을 수 있음  
        -> **형식** : 따옴표(“ ”) 같은 특수 문자, 태그 문자열의 형식 차이, CSV로 저장하며 바뀌는 자료형  
        -> CSV 저장 성공은 "파일을 쓰고 다시 읽을 수 있다"는 뜻일 뿐이라, 이런 품질 확인은 다음 정제 단계에서 따로 확인해 보아야 함

---

## 실습 마무리

**Q1. 이번 실습에서 사용한 HTML 수집의 전체 흐름은 무엇인가요?**

- 답변: HTML 요청 -> HTTP 오류 확인 -> beautifulsoup 파싱 -> 명언 영역 탐색 -> 영역별 필드 추출  
-> 필요한 경우 Next 링크를 따라 반복 -> dataframe -> CSV 저장

**Q2. HTTP 요청 성공과 원하는 HTML 데이터 추출 성공을 구분하는 이유는 무엇인가요?**

- 답변: 상태 코드 200이어도 초기 HTML이 원하는 요소가 없거나 선택자가 변경된 구조와 맞지 않을 수 있음  

**Q3. 같은 명언의 본문·작성자·태그를 연결하기 위해 어떤 기준을 사용했나요?**

- 답변: div.quote 하나를 명언 한 개의 경계로 삼고, 그 영역 안에서 span.text, small.author, div.tag, a.tag를 찾아  
하나의 dictionary로 구성했다.

**Q4. 다음 페이지로 이동하는 방법과 정상 종료 조건은 무엇인가요?**

- 답변: 현재 명언을 누적한 뒤 li.next a를 찾아 href를 꺼내고 url join으로 다음 url을 만든다.  
정상 처리한 현재 페이지에 Next 링크가 없으면 url을 None으로 만들어 종료한다.  

**Q5. CSS 선택자·XPath, 정적·동적 크롤링은 각각 무엇을 비교하는 개념인가요?**

- 답변: CSS 선택자와 XPath는 HTML의 원하는 요소를 찾는 표현방식, 정적 또는 동적 크롤링은 초기 HTML을 분석하는지, javascript실행이나 사용자 동작 후의 결과가 필요한지와 관련된 수집 방법

**Q6. API·HTML 크롤링 중 방법을 선택할 때 무엇을 확인하고, CSV 저장 후에는 무엇을 더 확인해야 하나요?**

- 답변: 필요한 필드와 수집 목적을 정한뒤 API 활용 가능 여부, 초기 HTML의 데이터 존재  
- script.network등 데이터 제공 방식, 브라우저 실행 필요성, 접근 정책을 확인한다.


### 제출 전 확인

- 필수 1·2와 과제 1의 코드·실행 결과·관찰표·비교표를 작성하세요.
- 각 해석 질문과 실습 마무리 질문에 빠짐없이 답하세요.
- 실제 실행 결과와 문제의 가정 사례(10+10+5)를 구분하세요.
- 실패했다면 오류 상황을 기록하고, 예전 CSV를 이번 실행 결과로 혼동하지 마세요.
- 작성한 `.ipynb`와 정상 생성한 `quotes_html_raw.csv`, `quotes_html_page2.csv`를 제출하세요.


## 개념과 결과를 함께 정리하기

| 구분할 내용 | 이번 강의의 기준 |
|---|---|
| 요청 실패 / 요소 0개 | 정상 응답을 받지 못한 상황 / 받은 HTML에서 해당 요소를 못 찾은 관찰 |
| 초기 HTML / 실행 후 화면 | 서버에서 받은 내용 / 브라우저가 JavaScript 등을 실행한 뒤 표시한 결과 |
| 요소 / 텍스트 | select_one()으로 찾은 구성 단위 / get_text()로 꺼낸 글자 |
| None / 빈 문자열 | 요소를 못 찾음 / 요소의 추출 글자가 비어 있음 |
| Next 글자 / href | 링크가 표시하는 글자 / 다음 요청에 사용할 주소 |
| urljoin() / requests.get() | 주소 구성 / 실제 요청 |
| 페이지 수 / 명언 수 | 처리한 페이지 횟수 / 누적 레코드 수 |
| API·HTML의 차이 / 공통점 | 원본 접근·다음 페이지 판단 방식 / 네 필드로 정리·저장 |
| 기본 저장 확인 / 품질 정제 | 행 수·컬럼·미리보기 확인 / 이후 중복·결측·누락 등의 검증 |

### 막혔을 때 먼저 볼 부분

| 현상 | 확인할 것 |
|---|---|
| 요청 실패 메시지 또는 HTTP 오류 | 주소·경로·연결·응답 상태 |
| 200인데 명언 영역이 없음 | 초기 HTML의 실제 구조와 선택자, 데이터 제공 방식 |
| 본문·작성자 요소가 None | 현재 명언 영역 안의 태그와 클래스 이름 |
| 다음 페이지로 이동하지 못함 | Next 요소의 존재와 href 값을 각각 확인 |
| 매번 마지막 페이지 데이터만 남음 | 레코드 목록을 while 안에서 다시 비우고 있지 않은지 |
| 상한 초과 오류 | Next가 예상대로 끝나는지, 반복한 주소와 처리 횟수 |
| CSV만 남아 있고 이번 실행은 오류 | 이전 파일인지, 이번 코드가 어느 줄에서 중단됐는지 |

학습 후에는 **어떤 HTML에서 어떤 영역을 골랐는지, 값들을 어떻게 같은 명언으로 연결했는지, 다음 페이지와 완료를 무엇으로 판단했는지, 왜 그 수집 방법을 골랐는지** 자신의 말로 설명해 보세요.
